# DDS-SLAM — Exact Paper Environment

Reproduces the exact environment from the DDS-SLAM paper:
- **Python 3.7** | **PyTorch 1.10.1+cu113** | **CUDA 11.3**
- **tinycudann v1.5** (commit `91ee479`) | **PyTorch3D v0.7.2**

First run takes ~20 min (downloads + compilation). Subsequent runs restore from cache (~5 min).

**Prerequisites**: Select a GPU runtime (Runtime → Change runtime type → T4 GPU)

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/DDS-SLAM/DDS-SLAM'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/bwright000/DDS-SLAM.git /content/DDS-SLAM
    # Optionally check out the upstream-matching branch for exact paper code:
    # !cd /content/DDS-SLAM && git checkout exact-paper-env
else:
    print(f'Repo already cloned at {REPO_DIR}')

os.chdir(REPO_DIR)
!pwd

## 2. Install Exact Paper Environment

This installs Python 3.7, CUDA 11.3 toolkit, PyTorch 1.10.1, tinycudann, pytorch3d, and all dependencies.

- First run: ~20 minutes
- From cache: ~5 minutes

In [ ]:
!bash Addons/colab_exact_env.sh --skip-data

## 3. Activate Environment

Set up environment variables for this notebook session.

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.3'
os.environ['PATH'] = '/usr/local/cuda-11.3/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.3/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# Use the Python 3.7 venv for all subsequent ! commands
PYTHON = '/tmp/dds_env/bin/python'
print(f'Python binary: {PYTHON}')
!{PYTHON} --version

## 4. Verify Environment

In [ ]:
PYTHON = '/tmp/dds_env/bin/python'

!{PYTHON} -c "
import sys
print(f'Python:       {sys.version}')

import torch
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA (torch): {torch.version.cuda}')
print(f'GPU:          {torch.cuda.get_device_name(0)}')
print(f'GPU avail:    {torch.cuda.is_available()}')

import numpy as np
print(f'NumPy:        {np.__version__}')

import tinycudann as tcnn
print(f'tinycudann:   OK')

from pytorch3d.transforms import matrix_to_quaternion
print(f'pytorch3d:    OK')

import marching_cubes
print(f'marching_cubes: OK')

import mathutils
print(f'mathutils:    OK')

# TCNN functional test
e = tcnn.Encoding(3, {
    'otype': 'HashGrid', 'n_levels': 1, 'n_features_per_level': 2,
    'log2_hashmap_size': 4, 'base_resolution': 4, 'per_level_scale': 1.0
})
x = torch.rand(1, 3, device='cuda')
y = e(x)
print(f'TCNN test:    OK (output shape {y.shape})')
print()
print('All checks passed!')
"

## 5. Upload / Download Dataset

Upload your dataset to `data/` or download it.

In [ ]:
import os
PYTHON = '/tmp/dds_env/bin/python'
REPO_DIR = '/content/DDS-SLAM/DDS-SLAM'
DATA_DIR = os.path.join(REPO_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# Option A: Copy from Google Drive (if you've uploaded data there)
# !cp -r /content/drive/MyDrive/DDS-SLAM-Data/Super {DATA_DIR}/Super

# Option B: Download Semantic-Super from Google Drive link
# !pip install -q gdown
# !gdown --id 1ZxWw2kNmgeMhBXAGyovL2icXHzn2OCVV -O {DATA_DIR}/Super.zip
# !unzip -q {DATA_DIR}/Super.zip -d {DATA_DIR} && rm {DATA_DIR}/Super.zip

# Check what's available
!ls -la {DATA_DIR}/ 2>/dev/null || echo 'No data directory yet'

## 6. Generate Depth Maps (if needed)

The paper used finetuned Monodepth2. Since those weights are unavailable, use one of the alternatives below.

In [ ]:
PYTHON = '/tmp/dds_env/bin/python'

# NOTE: generate_depth.py may need additional deps (transformers, etc.)
# Install them in the venv if needed:
# !{PYTHON} -m pip install transformers

# Generate depth maps for Semantic-Super trail3
# !{PYTHON} Addons/generate_depth.py --datadir data/Super --method depth_anything

print('Uncomment the commands above to generate depth maps')

## 7. Run DDS-SLAM

In [ ]:
PYTHON = '/tmp/dds_env/bin/python'

# Semantic-Super trail3 (151 frames, ~15 min)
!cd /content/DDS-SLAM/DDS-SLAM && {PYTHON} ddsslam.py --config ./configs/Super/trail3.yaml

## 8. Evaluate Results

In [ ]:
PYTHON = '/tmp/dds_env/bin/python'

# Evaluate rendering quality (PSNR, SSIM, LPIPS)
!{PYTHON} Addons/eval_rendering.py \
    --gt_dir data/Super/rgb \
    --render_dir output/trail3/demo \
    --name "Exact Paper Env" \
    --output_csv output/trail3/metrics_exact_env.csv \
    --sequence "Lab1 (trail3)"

In [ ]:
# Compare against paper and modern-stack results
print('Paper results:        PSNR=28.649  SSIM=0.797  LPIPS=0.231')
print('Modern stack (DA V2): PSNR=27.605  SSIM=0.754  LPIPS=0.372')
print()
print('Exact env results:')
!cat output/trail3/metrics_exact_env.csv 2>/dev/null || echo 'Run the evaluation cell above first'

## 9. Save Results to Google Drive

In [ ]:
import shutil, os

RESULTS_SRC = '/content/DDS-SLAM/DDS-SLAM/output'
RESULTS_DST = '/content/drive/MyDrive/DDS-SLAM-Results/exact_env'

if os.path.exists(RESULTS_SRC):
    os.makedirs(RESULTS_DST, exist_ok=True)
    shutil.copytree(RESULTS_SRC, RESULTS_DST, dirs_exist_ok=True)
    print(f'Results saved to {RESULTS_DST}')
else:
    print('No output directory found. Run DDS-SLAM first.')